# Tutorial 5: Doc-to-LoRA — Single-Pass Knowledge Internalization

**Turn a document into model weights in one forward pass, no gradient descent required.**

All previous strategies either avoid changing the model (JitRL, ACE) or use iterative gradient
descent (TTT-E2E). Doc-to-LoRA takes a fundamentally different approach: a **hypernetwork**
reads the document and directly outputs LoRA adapter weights in a single forward pass.

This is inspired by Sakana AI's research on hypernetwork-generated adapters, where a
"network that generates other networks' weights" can produce task-specific LoRA matrices
without any iterative optimization.

**In this tutorial you will learn:**
1. What LoRA is and why it is parameter-efficient
2. How a hypernetwork generates LoRA weights from text
3. Document chunking and rank concatenation
4. The LoRA injection/removal mechanism
5. How to measure forgetting before and after injection

## Concept: What is LoRA (Low-Rank Adaptation)?

LoRA (Low-Rank Adaptation) modifies a pretrained weight matrix **W** by adding a low-rank
decomposition instead of updating all parameters:

```
W' = W + A x B
```

where:
- **W** is the original weight matrix of shape `(d_out, d_in)` — frozen
- **A** is shape `(d_in, r)` — the "down-projection"
- **B** is shape `(r, d_out)` — the "up-projection"
- **r** is the LoRA rank, typically 4-64 (much smaller than d_in or d_out)

**Parameter savings**: Instead of storing `d_in x d_out` parameters for a full weight update,
LoRA only needs `d_in x r + r x d_out = r x (d_in + d_out)` parameters. For a 2048x2048
matrix with rank 8, that is `2 x 2048 x 8 = 32,768` vs. `2048 x 2048 = 4,194,304` — a **128x reduction**.

The key insight: weight updates during fine-tuning often have low intrinsic rank, so a
low-rank approximation captures most of the useful adaptation.

In [ ]:
# Cell 3: LoRA math demo — create matrices, show parameter savings
import torch

# Dimensions typical of a transformer MLP layer
d_in = 2048   # hidden dimension
d_out = 512   # intermediate dimension (simplified)
rank = 8      # LoRA rank

# Full weight update: d_in x d_out parameters
W = torch.randn(d_out, d_in)  # Original weight matrix
full_update = torch.randn(d_out, d_in)  # A full fine-tuning update

# LoRA: two small matrices
A = torch.randn(d_in, rank) * 0.01   # Down-projection
B = torch.randn(rank, d_out) * 0.01  # Up-projection
lora_update = A @ B  # Shape: (d_in, d_out) — same as a full update!

# Apply LoRA
W_prime = W + lora_update.T  # W' = W + (A x B)^T

full_params = d_out * d_in
lora_params = d_in * rank + rank * d_out

print("LoRA Parameter Efficiency Demo")
print("=" * 50)
print(f"Weight matrix shape: ({d_out}, {d_in})")
print(f"LoRA rank: {rank}")
print(f"A shape: ({d_in}, {rank})")
print(f"B shape: ({rank}, {d_out})")
print(f"A @ B shape: {lora_update.shape} (same as full update)")
print(f"")
print(f"Full update parameters:  {full_params:>12,}")
print(f"LoRA parameters:         {lora_params:>12,}")
print(f"Reduction factor:        {full_params / lora_params:>12.1f}x")
print(f"")
print(f"W  norm: {W.norm():.4f}")
print(f"Delta norm: {lora_update.norm():.4f}")
print(f"W' norm: {W_prime.norm():.4f}")
print(f"Relative change: {lora_update.norm() / W.norm() * 100:.4f}%")

In [ ]:
# Cell 4: LoRA injection demo — add delta to a linear layer, show weight change
import torch.nn as nn

# Create a simple linear layer (simulating one MLP layer)
layer = nn.Linear(64, 32, bias=False)
original_weight = layer.weight.data.clone()

# Create LoRA matrices
r = 4
A = torch.randn(64, r) * 0.01
B = torch.randn(r, 32) * 0.01
delta = (A @ B).T  # (out_features=32, in_features=64)

# Inject: W' = W + delta
layer.weight.data += delta

print("LoRA Injection Demo")
print("=" * 50)
print(f"Layer shape: {layer.weight.shape}")
print(f"Original weight norm: {original_weight.norm():.4f}")
print(f"Delta norm: {delta.norm():.4f}")
print(f"Modified weight norm: {layer.weight.data.norm():.4f}")
print(f"Weight changed: {not torch.equal(original_weight, layer.weight.data)}")

# Remove: restore original
layer.weight.data = original_weight
print(f"After removal, weight matches original: {torch.equal(original_weight, layer.weight.data)}")

## Concept: The Hypernetwork — A Network That Generates Other Networks' Weights

A hypernetwork is a neural network whose output is the **weights** of another network.
In Doc-to-LoRA, the hypernetwork reads a document and outputs LoRA A and B matrices
for every target layer in the base model.

The real architecture uses a **Perceiver** with cross-attention:
1. Document text is tokenized and run through the frozen base model
2. Hidden-state activations become the Perceiver's **context** (keys/values)
3. Learned **latent queries** (one per target layer) attend to the context
4. Output heads project latents to LoRA A and B matrices

For this tutorial, we use `SimulatedHypernetwork` — a deterministic hash-based
simulator that produces small random LoRA matrices seeded by the document text.
This lets us test the full pipeline without a pretrained Perceiver checkpoint.

In [ ]:
# Cell 6: SimulatedHypernetwork demo — generate LoRA from text, inspect shapes
from continual_learning.doc2lora.hypernetwork import SimulatedHypernetwork

hypernet = SimulatedHypernetwork(
    hidden_dim=2048,
    num_layers=4,        # Targeting 4 layers (reduced for demo)
    lora_rank=8,
    intermediate_dim=512,
)

# Generate LoRA from a single chunk
single_chunk = ["Oracle Labs established its Quantum Computing Research Division in 2019."]
weights_single = hypernet.generate_lora(single_chunk)

print("Single-Chunk LoRA Generation")
print("=" * 50)
for key, tensor in sorted(weights_single.items()):
    print(f"  {key}: shape={tensor.shape}, norm={tensor.norm():.4f}")

# Generate LoRA from multiple chunks — rank concatenation
multi_chunks = [
    "Oracle Labs established its Quantum Computing Research Division in 2019.",
    "The RedShift processor achieved quantum volume 512.",
    "Lattice Shield was integrated into OCI key management.",
]
weights_multi = hypernet.generate_lora(multi_chunks)

print(f"\nMulti-Chunk LoRA Generation ({len(multi_chunks)} chunks)")
print("=" * 50)
for key, tensor in sorted(weights_multi.items()):
    print(f"  {key}: shape={tensor.shape}, norm={tensor.norm():.4f}")

print(f"\nEffective rank: single={hypernet.lora_rank}, multi={hypernet.lora_rank * len(multi_chunks)}")
print("Note: multi-chunk A matrices are wider (rank concatenation along dim=1)")
print("Note: multi-chunk B matrices are taller (rank concatenation along dim=0)")

## Concept: Document Chunking

Long documents are split into fixed-size chunks (default: 1024 words) before being processed
by the hypernetwork. Each chunk produces a rank-r LoRA, and they are composed via **rank concatenation**:

```
Chunk 1  -->  A1 (d, r),  B1 (r, d')      Effective rank = r
Chunk 2  -->  A2 (d, r),  B2 (r, d')      Effective rank = r
Chunk 3  -->  A3 (d, r),  B3 (r, d')      Effective rank = r
                    |
            Concatenate along rank dimension:
            A_final = [A1 | A2 | A3]  shape (d, 3r)
            B_final = [B1 ; B2 ; B3]  shape (3r, d')
                    |
            A_final @ B_final = same shape as single-chunk
            but with effective rank = 3r
```

Two modes:
- **"doc" mode**: splits into 1024-word chunks, each gets a rank-r LoRA
- **"text" mode**: treats the entire input as a single chunk (for task descriptions)

In [ ]:
# Cell 8: DocumentChunker demo — chunk the sample document
from pathlib import Path
from continual_learning.doc2lora.chunker import DocumentChunker

document = Path("data/sample_document.txt").read_text()

# Default chunk size (1024 words)
chunker_default = DocumentChunker(chunk_size=1024)
chunks_default = chunker_default.chunk(document, mode="doc")

# Smaller chunk size to see splitting in action
chunker_small = DocumentChunker(chunk_size=100)
chunks_small = chunker_small.chunk(document, mode="doc")

# Text mode (single chunk regardless of length)
chunks_text = chunker_default.chunk(document, mode="text")

print(f"Document: {len(document.split())} words")
print(f"")
print(f"Mode='doc', chunk_size=1024: {len(chunks_default)} chunk(s)")
for i, c in enumerate(chunks_default):
    print(f"  Chunk {i}: {len(c.split())} words — '{c[:60]}...'")

print(f"\nMode='doc', chunk_size=100: {len(chunks_small)} chunk(s)")
for i, c in enumerate(chunks_small):
    print(f"  Chunk {i}: {len(c.split())} words — '{c[:60]}...'")

print(f"\nMode='text' (single chunk): {len(chunks_text)} chunk(s)")
print(f"  Chunk 0: {len(chunks_text[0].split())} words")

In [ ]:
# Cell 9: Create Doc2LoRAEngine in "doc" mode, learn document
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float32, trust_remote_code=True
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print(f"Model loaded on {device}")

from continual_learning.doc2lora.engine import Doc2LoRAEngine

doc_engine = Doc2LoRAEngine(
    model=model,
    tokenizer=tokenizer,
    hidden_dim=2048,
    num_target_layers=4,    # Fewer layers for demo speed
    intermediate_dim=512,
    lora_rank=8,
    chunk_size=1024,
    mode="doc",
    simulated=True,         # Use SimulatedHypernetwork
    layer_prefix="model.layers.{i}.mlp",
)

# Learn the document
def learn_callback(**kwargs):
    print(f"  Callback: {kwargs}")

print(f"\nLearning document in 'doc' mode...")
t0 = time.time()
result = doc_engine.learn(document, callback=learn_callback)
learn_time = time.time() - t0

print(f"\nResult:")
print(f"  Method: {result['method']}")
print(f"  Mode: {result['mode']}")
print(f"  Chunks: {result['num_chunks']}")
print(f"  Effective rank: {result['effective_rank']} (base rank {doc_engine._hypernetwork.lora_rank} x {result['num_chunks']} chunks)")
print(f"  Tokens processed: {result['tokens_processed']}")
print(f"  Time: {learn_time:.3f}s")
print(f"  LoRA injected: {doc_engine._injector.is_injected}")

In [ ]:
# Cell 10: Query about the document
questions = [
    "Who led the Quantum Computing Research Division at Oracle Labs?",
    "What was the RedShift processor's quantum volume?",
    "When was Lattice Shield integrated into OCI?",
]

print("Querying the LoRA-adapted model")
print("=" * 60)
for q in questions:
    response = doc_engine.generate(q, max_new_tokens=100)
    print(f"\nQ: {q}")
    print(f"A: {response[:300]}")

print(f"\nEffective rank of injected LoRA: {doc_engine._hypernetwork.lora_rank * result['num_chunks']}")
print(f"Number of modified layers: {len(doc_engine._injector._original_weights)}")

In [ ]:
# Cell 11: Switch to "text" mode — task-specialized LoRA
doc_engine.clear()  # Remove previous LoRA
print(f"LoRA injected after clear: {doc_engine._injector.is_injected}")

# Switch to text mode
doc_engine.set_mode("text")
print(f"Mode: {doc_engine.mode}")

# In text mode, the entire input is treated as a single chunk (task description)
task_description = """You are a helpful assistant that specializes in answering questions about
quantum computing research, focusing on Oracle Labs' contributions to the field including
their RedShift processor, QubitFlow framework, and Lattice Shield encryption protocol."""

result_text = doc_engine.learn(task_description)
print(f"\nText mode result:")
print(f"  Chunks: {result_text['num_chunks']} (always 1 in text mode)")
print(f"  Effective rank: {result_text['effective_rank']}")
print(f"  Mode: {result_text['mode']}")

# Query with text-mode LoRA
response = doc_engine.generate("What is quantum computing?", max_new_tokens=100)
print(f"\nQ: What is quantum computing?")
print(f"A: {response[:300]}")

# Switch back to doc mode for remaining demos
doc_engine.clear()
doc_engine.set_mode("doc")

In [ ]:
# Cell 12: LoRAInjector internals — inject/remove, verify restoration
from continual_learning.doc2lora.lora_injector import LoRAInjector
from continual_learning.doc2lora.hypernetwork import SimulatedHypernetwork

injector = LoRAInjector()

# Generate some LoRA weights
hypernet = SimulatedHypernetwork(
    hidden_dim=model.config.hidden_size,
    num_layers=2,
    lora_rank=8,
    intermediate_dim=512,
)
lora_weights = hypernet.generate_lora(["test document chunk"])

# Capture weights before injection
target_module_name = "model.layers.0.mlp"
target_module = dict(model.named_modules()).get(target_module_name)

if target_module is not None and isinstance(target_module, torch.nn.Linear):
    weight_before = target_module.weight.data.clone()
    print(f"Target: {target_module_name}")
    print(f"Weight shape: {target_module.weight.shape}")
    print(f"Weight norm before: {weight_before.norm():.6f}")

    # Inject LoRA
    injector.inject(model, lora_weights, layer_prefix="model.layers.{i}.mlp")
    print(f"\nAfter injection:")
    print(f"  is_injected: {injector.is_injected}")
    print(f"  Weight norm: {target_module.weight.data.norm():.6f}")
    print(f"  Weight changed: {not torch.equal(weight_before, target_module.weight.data)}")
    print(f"  Stored originals: {list(injector._original_weights.keys())}")

    # Remove LoRA — should restore exact original weights
    injector.remove(model)
    print(f"\nAfter removal:")
    print(f"  is_injected: {injector.is_injected}")
    print(f"  Weight norm: {target_module.weight.data.norm():.6f}")
    print(f"  Weights restored: {torch.equal(weight_before, target_module.weight.data)}")
else:
    print(f"Module '{target_module_name}' not found or not a Linear layer.")
    print("This is expected if the model uses a different architecture.")
    print("The injector silently skips non-matching layers.")
    print(f"\nAvailable layer patterns (first 5):")
    for name, mod in list(model.named_modules())[:20]:
        if isinstance(mod, torch.nn.Linear):
            print(f"  {name}: {mod.weight.shape}")

## Forgetting Analysis

A critical question for any continual learning strategy: **does learning new knowledge
cause the model to forget old knowledge?**

With Doc-to-LoRA, we can measure this directly:
1. Test the base model on holdout questions (baseline accuracy)
2. Inject LoRA for a new document
3. Re-test the same holdout questions (post-injection accuracy)
4. Compute forgetting ratio: `(before - after) / before`

A forgetting ratio of:
- **0.0** = no forgetting (perfect)
- **0.5** = lost half the original knowledge
- **1.0** = total catastrophic forgetting
- **negative** = the LoRA actually improved performance (possible if the adaptation is beneficial)

In [ ]:
# Cell 14: Measure forgetting before/after LoRA injection
from continual_learning.evaluation.forgetting_metrics import compute_forgetting_ratio
from continual_learning.evaluation.benchmarks import evaluate_qa_accuracy

# Holdout questions about general knowledge (not from the document)
holdout_items = [
    {"question": "What is the capital of France?", "answer": "Paris"},
    {"question": "What is 2 + 2?", "answer": "4"},
    {"question": "What planet is closest to the sun?", "answer": "Mercury"},
    {"question": "What is the chemical symbol for water?", "answer": "H2O"},
    {"question": "Who wrote Romeo and Juliet?", "answer": "Shakespeare"},
]

# Step 1: Baseline accuracy (no LoRA)
print("Step 1: Measuring baseline accuracy (no LoRA)...")
baseline_accuracy = evaluate_qa_accuracy(model, tokenizer, holdout_items)
print(f"  Baseline accuracy: {baseline_accuracy:.2%}")

# Step 2: Inject LoRA for the quantum computing document
print("\nStep 2: Injecting document LoRA...")
doc_engine_fresh = Doc2LoRAEngine(
    model=model, tokenizer=tokenizer,
    hidden_dim=2048, num_target_layers=4,
    intermediate_dim=512, lora_rank=8,
    mode="doc", simulated=True,
    layer_prefix="model.layers.{i}.mlp",
)
doc_engine_fresh.learn(document)
print(f"  LoRA injected: {doc_engine_fresh._injector.is_injected}")

# Step 3: Re-test holdout questions
print("\nStep 3: Measuring post-injection accuracy...")
post_accuracy = evaluate_qa_accuracy(model, tokenizer, holdout_items)
print(f"  Post-injection accuracy: {post_accuracy:.2%}")

# Step 4: Compute forgetting
forgetting = compute_forgetting_ratio(baseline_accuracy, post_accuracy)
print(f"\nForgetting ratio: {forgetting:.4f}")
if forgetting <= 0:
    print("  Interpretation: No forgetting (or improvement)!")
elif forgetting < 0.1:
    print("  Interpretation: Minimal forgetting — LoRA is well-contained.")
elif forgetting < 0.3:
    print("  Interpretation: Moderate forgetting — consider lower rank or fewer layers.")
else:
    print("  Interpretation: Significant forgetting — reduce LoRA strength.")

# Clean up: remove LoRA to restore original model
doc_engine_fresh.clear()
print(f"\nLoRA removed. Model restored to original state.")

## Exercises

1. **Rank sweep**: Create engines with `lora_rank` of 2, 4, 8, 16, and 32. For each,
   learn the same document and measure both document QA accuracy and forgetting ratio.
   What is the sweet spot where you get good knowledge retention without excessive forgetting?

2. **Layer targeting**: Compare targeting 4 layers vs. 8 layers vs. all layers.
   Does injecting into more layers improve document QA? Does it increase forgetting?

3. **Chunk size experiment**: Try chunk sizes of 256, 512, 1024, and 2048 words.
   Smaller chunks mean higher effective rank (more chunks). Does this help or hurt?

4. **Doc vs. Text mode**: Learn the same content as a document (doc mode) and as a
   task description (text mode). Compare the generated responses. When is each mode better?

5. **Compare with TTT-E2E**: Learn the same document with both Doc-to-LoRA and TTT-E2E.
   Measure accuracy and forgetting for each. Doc-to-LoRA should be faster (single pass)
   but TTT-E2E may internalize knowledge more deeply (iterative gradient descent).